In [1]:
import numpy as np

aliases = ['long', 'ulong', 'longlong', 'ulonglong', 'float96', 'float128', 'complex192', 'complex256']
for alias in aliases:
    if not hasattr(np, alias):
        setattr(np, alias, int if 'int' in alias or 'long' in alias else float)

import syft as sy
import torch as th
from sklearn import datasets
import pandas as pd

/tmp/ipykernel_4596/3551385867.py:5: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, alias):
/tmp/ipykernel_4596/3551385867.py:5: FutureWarning: In the future `np.ulong` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, alias):
/home/malak/my files/f1/tutorial-env/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=2.0.0 and <2.8.0 is required for this version of SciPy (detected version 1.26.4)
  from scipy.sparse import csr_array, issparse


## 1. Node Initialization & Authentication (Hospital A)
Spinning up an isolated PySyft Domain Node for Hospital A using `sy.orchestra` and authenticating as Domain Admin to initialize the node environment.

In [ ]:
node = sy.orchestra.launch(name ="heart_disease", port=1110 ,reset=True)
hosp1 =node.login(email ="info@openmined.org",password="changethis")


Starting heart_disease server on 0.0.0.0:1110


INFO:     Starting server with settings: {'name': 'heart_disease', 'server_type': <ServerType.DATASITE: 'datasite'>, 'server_side_type': <ServerSideType.HIGH_SIDE: 'high'>, 'deployment_type': <DeploymentType.PYTHON: 'python'>, 'processes': 1, 'reset': True, 'dev_mode': False, 'enable_warnings': False, 'in_memory_workers': True, 'queue_port': None, 'create_producer': False, 'n_consumers': 0, 'association_request_auto_approval': False, 'background_tasks': False, 'db_config': None, 'db_url': None} and worker class: <class 'syft.server.datasite.Datasite'>
CRITICAL:syft.server.server:Hash of the signing key '9ce63...'


Using SQLiteDBConfig and sqlite:////tmp/syft/b1098354f807440484a04a9a7e161e14/db/b1098354f807440484a04a9a7e161e14_json.db


INFO:     Started server process [4726]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:1110 (Press CTRL+C to quit)


Waiting for server to start Done.


SyftInfo: You have launched a development server at http://0.0.0.0:1110. It is intended only for local use.

Logged into <heart_disease: High side Datasite> as <info@openmined.org>


SyftWarning: You are using a default password. Please change the password using `[your_client].account.set_password([new_password])`.

Error: <class 'syft.service.code.user_code.UserCodeStatusCollection'> Your code is waiting for approval. Code status on server 'heart_disease' is 'UserCodeStatus.PENDING'.
Traceback (most recent call last):
  File "/home/malak/my files/f1/tutorial-env/lib/python3.12/site-packages/syft/server/server.py", line 1186, in handle_api_call_with_unsigned_result
    result = method(context, *api_call.args, **api_call.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/malak/my files/f1/tutorial-env/lib/python3.12/site-packages/syft/service/service.py", line 490, in _decorator
    result = func(self, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/malak/my files/f1/tutorial-env/lib/python3.12/site-packages/syft/service/code/user_code_service.py", line 423, in call
    return self._call(context, uid, **kwargs).unwrap()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/malak/my files/f1/tutorial-env/lib/python3.12/site-pa

## 2. User Provisioning & Role Assignment (Hospital A)
Creating a Data Scientist user account on Hospital A's node with restricted `DATA_SCIENTIST` permissions to enable remote computation requests.

In [3]:
from syft.service.user.user_roles import ServiceRole
hosp1.users.create(
    name="malak_scientist",
    email="malak@hospascientist.org",
    password="1234",
    role =ServiceRole.DATA_SCIENTIST
)


```python
class UserView:
  id: str = 46ca1a5738d442dea19c2b5e6d803a74
  name: str = "malak_scientist"
  email: str = "malak@hospascientist.org"
  institution: str = ""
  website: str = ""
  role: str = ServiceRole.DATA_SCIENTIST
  notifications_enabled: str = {: True, : False, : False, : False}

```

## 3. Local Dataset Ingestion (Hospital A)
Loading raw healthcare record data (`heart_h1.csv`) into memory on Hospital A's local node environment prior to preprocessing and mock dataset creation.

In [4]:
df = pd.read_csv("heart_h1.csv")
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,62,0,0,124,209,0,1,163,0,0.0,2,0,2,1
1,53,0,2,128,216,0,0,115,0,0.0,2,0,0,1
2,55,1,0,160,289,0,0,145,1,0.8,1,1,3,0
3,50,0,1,120,244,0,1,162,0,1.1,2,0,2,1
4,48,1,0,130,256,1,0,150,1,0.0,2,2,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,66,0,2,146,278,0,0,152,0,0.0,1,1,2,1
457,65,1,3,138,282,1,0,174,0,1.4,1,1,2,0
458,39,0,2,138,220,0,1,152,0,0.0,1,0,2,1
459,71,0,2,110,265,1,0,130,0,0.0,2,1,2,1


In [5]:
x = df.drop('target', axis=1)   
y = df['target']


In [6]:
x = th.tensor(x.values, dtype=th.float32)
y  = th.tensor(y.values, dtype=th.long)
type(y), type(x)


(torch.Tensor, torch.Tensor)

In [8]:
heart_asset_data = sy.Asset(
    name="heart_disease_hospital_A_data",
    description="Heart disease dataset",
    data = x,
    mock = th.zeros(x.shape).float()
)
heart_asset_target = sy.Asset(
    name="heart_disease_hospital_A_target",
    description="Heart disease dataset target",
    data = y,
    mock = th.zeros(y.shape).long()
)


In [9]:
heart_disease_dataset= sy.Dataset(
    name ="heart_disease_dataset",
    description= "data set of hospital 1",
    asset_list=[heart_asset_data,heart_asset_target]
)


hosp1.upload_dataset(heart_disease_dataset)


Uploading: heart_disease_hospital_A_target: 100%|██████████| 2/2 [00:00<00:00,  8.81it/s]


SyftSuccess: Dataset uploaded to 'heart_disease'. To see the datasets uploaded by a client on this server, use command `[your_client].datasets`

## 4. Node Admin Request Review (Hospital A)
Inspecting incoming execution requests submitted by the Data Scientist to review code details and verify the initial pending status (`PENDING`).

In [11]:
req =hosp1.requests[0]
print(req.status)

RequestStatus.PENDING


In [12]:
req.code

```python
@sy.syft_function(
    input_policy=sy.ExactMatch(
        heart_disease_data=heart_disease_hospital_a_data,
        heart_disease_target=heart_disease_hospital_a_target,
    ),
)
def train_model_h1(heart_disease_data, heart_disease_target):
    import copy
    import torch as th
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    scaled_x = scaler.fit_transform(heart_disease_data)

    x_train, x_val, y_train_split, y_val_split = train_test_split(
        scaled_x,
        heart_disease_target,
        test_size=0.2,
        random_state=42,
        stratify=heart_disease_target,
    )

    x_train_t = th.tensor(x_train, dtype=th.float32)
    y_train_t = th.tensor(y_train_split, dtype=th.long).squeeze()
    x_val_t = th.tensor(x_val, dtype=th.float32)
    y_val_t = th.tensor(y_val_split, dtype=th.long).squeeze()

    in_features = x_train_t.shape[1]
    model = th.nn.Sequential(
        th.nn.Linear(in_features, 64),
        th.nn.ReLU(),
        th.nn.Linear(64, 16),
        th.nn.LogSoftmax(dim=1),
    )

    optimizer = th.optim.Adam(params=model.parameters(), lr=0.001)

    patience = 15
    patience_counter = 0
    best_val_loss = float("inf")
    best_model_weights = None

    for epoch in range(150):
        model.train()
        optimizer.zero_grad()
        output = model(x_train_t)
        loss = th.nn.functional.nll_loss(output, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with th.no_grad():
            val_output = model(x_val_t)
            val_loss = th.nn.functional.nll_loss(
                val_output, y_val_t
            ).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return (
        best_model_weights
        if best_model_weights is not None
        else model.state_dict()
    )
```

In [13]:
req.approve()

Approving request on change train_model_h1 for datasite heart_disease


SyftSuccess: Request 94c993de5d56498b8ac907bcf2c9ab5f changes applied